# 04 · Modelagem preditiva

Protocolo fixado **antes** de qualquer modelo: partição por grupo (hash das
features), holdout de 20% tocado uma única vez, `class_weight` em vez de
reamostragem, **PR-AUC** como métrica principal.

> Documentos: [`docs/08`](../docs/08-modelagem-preditiva.md) e [`docs/10`](../docs/10-frente1-variaveis-expandidas.md)


In [1]:
import sys, json
from pathlib import Path

# a raiz e onde existe src/ — funciona rodando de notebooks/ ou da raiz do repo
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

GOLD = RAIZ / "data" / "processed" / "gold"
def ler(nome, base=GOLD):
    return json.loads((base / nome).read_text(encoding="utf-8"))


## A escada — cada degrau justificado


In [2]:
esc = ler("_escada_modelos.json")
linhas = [{"modelo": k, "vars": v["variaveis"],
           "PR-AUC": v["holdout"]["pr_auc"],
           "ganho vs prevalência": v["holdout"]["pr_auc_ganho"],
           "ROC-AUC": v["holdout"]["roc_auc"],
           "recall@esp90": v["holdout"]["recall_esp90"],
           "ECE": v["holdout"]["ece"]}
          for k, v in esc["sem_proxies_de_acesso"]["modelos"].items()]
pd.DataFrame(linhas)


,modelo,vars,PR-AUC,ganho vs prevalência,ROC-AUC,recall@esp90,ECE
0,0_prevalencia,18,0.1428,1.00,0.5000,1.0000,0.00112
1,1_regra_clinica,3,0.3350,2.35,0.7771,0.3645,0.00850
2,2_logistica_l2,18,0.4139,2.90,0.8252,0.4601,0.24558
3,3_spline,18,0.4402,3.08,0.8312,0.4792,0.24165
4,4_gradient_boosting,18,0.4494,3.15,0.8347,0.4926,0.23606
5,5_gb_calibrado,18,0.4504,3.15,0.8347,0.4904,0.00350


### O que a coluna ECE conta

`class_weight="balanced"` dá **ECE 0,236**; a calibração isotônica dá **0,0035** —
**67× menos erro** com ROC-AUC idêntico. Um modelo com peso de classe ordena bem
mas diz "40%" onde a prevalência é 14%.

É a demonstração empírica do ADR 0004: **reponderar destrói a calibração**. E o
mesmo argumento condena o SMOTE.


## Três previsões que os dados não confirmaram


In [3]:
v = esc["vazamento"]
print("1 · VAZAMENTO POR DUPLICATA")
print(f"   {v['pct_teste_contaminado']}% do teste tem gêmea idêntica no treino")
print(f"   inflação de PR-AUC: {v['inflacao_pr_auc_%']}%  <- muito menor que o alegado")
print()
t = esc["teto_de_bayes"]
print("2 · TETO DE BAYES")
print(f"   acerto máximo imposto pelo ruído: {t['acerto_maximo_ponderado']*100:.2f}%")
print(f"   ROC-AUC do melhor modelo: 0,836  ->  o teto NÃO é a restrição")
print()
sem = esc["sem_proxies_de_acesso"]["modelos"]["5_gb_calibrado"]["holdout"]["pr_auc"]
com = esc["com_proxies_de_acesso"]["modelos"]["5_gb_calibrado"]["holdout"]["pr_auc"]
print("3 · PROXIES DE ACESSO")
print(f"   sem: {sem}   com: {com}   ganho: {(com/sem-1)*100:.1f}%")
print("   excluí-los por validade quase não custa performance")


1 · VAZAMENTO POR DUPLICATA
   13.67% do teste tem gêmea idêntica no treino
   inflação de PR-AUC: 0.09%  <- muito menor que o alegado

2 · TETO DE BAYES
   acerto máximo imposto pelo ruído: 99.30%
   ROC-AUC do melhor modelo: 0,836  ->  o teto NÃO é a restrição

3 · PROXIES DE ACESSO
   sem: 0.4504   com: 0.4526   ganho: 0.5%
   excluí-los por validade quase não custa performance


As três estavam escritas em documento anterior deste projeto e foram **corrigidas
na fonte**. A restrição real não é ruído de rótulo nem algoritmo: é **informação**.


## Curva de parcimônia — quantas variáveis bastam


In [4]:
p = pd.DataFrame(esc["parcimonia"]["curva"])
p[["n_variaveis", "adicionada", "pr_auc", "%_do_teto"]]


,n_variaveis,adicionada,pr_auc,%_do_teto
0,1,saude_geral,0.2706,59.8
1,2,imc,0.3362,74.3
2,3,hipertensao,0.3780,83.5
3,4,colesterol_alto,0.3984,88.0
4,5,idade_faixa,0.4039,89.2
5,6,alcool_excessivo,0.4073,90.0
6,7,sexo,0.4095,90.5
7,8,doenca_cardiaca,0.4110,90.8


**Cinco variáveis entregam 89,2%** do modelo de 21. O entregável é o escore.


## E o que acontece ao recuperar as variáveis descartadas


In [5]:
f1 = ler("_frente1_expandido.json")
display(pd.DataFrame(f1["comparacao"]).T[["pr_auc", "roc_auc", "recall_esp90", "n_variaveis"]]
        .dropna().head(3))
print("\nDe onde vem o ganho (ablação por bloco):")
display(pd.DataFrame(f1["ablacao_por_bloco"])[["bloco_removido", "n_removidas", "perda_%"]])


,pr_auc,roc_auc,recall_esp90,n_variaveis
21_originais,0.4442,0.8432,0.5047,21.0
60_risco,0.4725,0.8530,0.5325,60.0
69_com_deteccao,0.4897,0.8633,0.5572,69.0



De onde vem o ganho (ablação por bloco):


,bloco_removido,n_removidas,perda_%
0,comorbidades,11,8.40
1,antropometria,3,6.07
2,saude_percebida,3,3.43
3,alcool,5,1.46
4,dieta,8,1.27
5,raca,2,0.76
6,demografia,11,0.68
7,limitacao_funcional,7,0.57
8,tabaco,3,0.11
9,atividade_fisica,6,0.06


## Para quem o modelo melhora — a auditoria que só agora é possível


In [6]:
aud = pd.DataFrame(f1["auditoria_raca"])
aud[["grupo", "n", "prevalencia_%", "recall_orig", "recall_novo", "ganho_recall"]]


,grupo,n,prevalencia_%,recall_orig,recall_novo,ganho_recall
0,branco nao-hispanico,65971,12.31,0.4935,0.4845,-0.0090
1,negro nao-hispanico,6687,20.95,0.5860,0.6938,0.1078
2,outro nao-hispanico,3858,14.52,0.4232,0.5446,0.1214
3,multirracial nao-hispanico,1590,13.65,0.5207,0.6590,0.1383
4,hispanico,7127,14.54,0.5193,0.6496,0.1303


> **O ganho médio de 6,6% esconde uma redistribuição enorme.** Brancos perdem
> meio ponto; todos os demais grupos ganham **10 a 13 pontos percentuais** de recall.
>
> O modelo de 21 variáveis era sistematicamente pior para minorias — e ninguém
> podia saber, porque a variável que revela isso tinha sido removida da base.
